## Example of Video Searching and Summarization Agent


#### This use case requires ffmpeg. Please first check whether it is already installed; otherwise, install FFmpeg (e.g., on Mac, run brew install ffmpeg).

In [ ]:
!which ffmpeg

In [ ]:
!pip install numpy==2.0.2

Please check whether the numpy verson is 2.0.x; this is required

In [ ]:
import numpy
print(numpy.__version__)

### Import Modules

In [ ]:
from slackagents import Assistant, FunctionTool, OpenAILLM, BaseLLMConfig

### Create Custom Tools
#### Youtube video searching tool
Allow agent to search relevant youtube videos based on the user's query.
#### Video summarization tool
For each returned video, the tool extracts its audio, transcribes it into text, and then summarize the video content based on the transcribed text.

1. Create video searching tool

In [ ]:
!pip install youtube-search-python

In [6]:
import json
from youtubesearchpython import VideosSearch

def video_search(
    query: str,
    max_results: int=3
) -> str:
    """A tool to search videos on youtube.com
    :param query: the query to be used for searching videos
    :type query: string
    :param max_results: The maximum number of results to return
    :type max_results: number
    :return: a string in json format
    :rtype: string
    """
    videos_search = VideosSearch(
        query=query, 
        limit=max_results, 
        region='US', 
        language='en'
    )
    results = videos_search.result()
    filtered_results = [
    {
        "title": video["title"],
        "link": video["link"],
        "duration": video["duration"],
        "views": video.get("viewCount", {}).get("text", "N/A")
    }
    for video in results["result"]
    ]
    filtered_results_json = json.dumps(filtered_results, indent=4)
    return filtered_results_json

Create tool from tool function

In [ ]:
video_search_tool = FunctionTool.from_function(video_search)
print(f"Video Search Tool: {json.dumps(video_search_tool.info, indent=4)}")

2. Create video summarization tool

In [ ]:
!pip install yt-dlp
!pip install librosa
!pip install soundfile

In [ ]:
import os
import yt_dlp
import librosa
from openai import OpenAI
import soundfile as sf


def summarize_youtube_video(
        youtube_url: str
        ) -> str:
    """A tool to summarize youtube video content
    :param youtube_url: The youtube video url to be summarized
    :type youtube_url: string
    :return: A text summary of the youtube video
    :rtype: string
    """
    client = OpenAI()
    output_dir = "audio_cache"  # temp dir to store video cache
    # Step 1: Download audio from YouTube
    audio_file = download_audio(youtube_url, output_dir)
    # Step 2: chunk audio
    chunk_audio(audio_file)
    # # Step 2: Transcribe the audio
    transcription = transcribe_audio(audio_file, client)
    # step 3: summarize video
    summary = summarize(transcription)
    os.remove(audio_file)
    return summary

def chunk_audio(audio_file, segment_length=600):
    audio, sr = librosa.load(audio_file, sr=44100)
    end = int(segment_length * sr)
    # Use the first chucnk for summarization for now.
    first_chunk = audio[:end]
    sf.write(audio_file, first_chunk, sr)
    print("chunk success!")
    return audio_file


def chunk_text(text, max_length=3000):
    sentences = text.split('. ')
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        sentence = sentence + '.'
        if len(current_chunk) + len(sentence) <= max_length:
            current_chunk += sentence
        else:
            chunks.append(current_chunk)
            current_chunk = sentence
    if current_chunk:
        chunks.append(current_chunk)
    return chunks


def summarize(transcription):
    chunks = chunk_text(transcription)
    # only summarize the first chunk for now
    llm = OpenAILLM(BaseLLMConfig(model="gpt-4o"))
    system_prompt = """
    You are a helpful assistant tasked with summarizing YouTube videos. 
    You will be provided with text transcribed from the video's audio. 
    Your task is to generate a summary of the video based on this transcribed text. Please make the summary less than 5 sentences.
    The summary should clearly convey the video's main content, and if possible, include bullet points highlighting key points.
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": chunks[0]}
    ]
    summary = llm.chat_completion(messages)["content"].strip()
    return summary


def transcribe_audio(file_path, client):
    audio_file = open(file_path, "rb")
    transcription = client.audio.transcriptions.create(model="whisper-1", file=audio_file)
    return transcription.text


def download_audio(youtube_url, output_dir="outputs"):
    a_type = "mp3"
    ydl_config = {
        "format": "bestaudio/best",
        "postprocessors": [
            {
                "key": "FFmpegExtractAudio",
                "preferredcodec": a_type,
                "preferredquality": "192"
            }
        ],
        "outtmpl": os.path.join(output_dir, "temp"),
        "verbose": True,
    }
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    print(f"Downloading video from {youtube_url}")

    try:
        with yt_dlp.YoutubeDL(ydl_config) as ydl:
            ydl.download([youtube_url])
    except:
        return False
    return os.path.join(output_dir, f"temp.{a_type}")


Create tool from tool function

In [ ]:
import json
video_summarization_tool = FunctionTool.from_function(summarize_youtube_video)
print(f"Video Summarization Tool: {json.dumps(video_summarization_tool.info, indent=4)}")

### Initialize ToolAssistant

In [9]:
assistant = Assistant(
    name="Video Assistant",
    desc="An assistant that can help search youtube video and return basic information and summarization of the videos",
    llm=OpenAILLM(BaseLLMConfig(model="gpt-4o")),
    tools=[video_search_tool, video_summarization_tool],
    system_prompt="You are an AI assistant that can help search youtube video and return a list of videos with basic information and summarization. If not specified, please generate two videos. Please always return the list of videos with title, youtube url, duration, number of views, and the generated summarization.",
    verbose=True # whether to print the assistant's tool request and response messages to the console
)

Test the agent

In [ ]:
from IPython.display import Markdown
message = "Can you help me search some videos about what is Salesforce?"
response = assistant.chat(message)
display(Markdown(response))